In [1]:
import copy
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import average_precision_score
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.model_selection import StratifiedKFold
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import lightning as L
from lightning.pytorch.callbacks import EarlyStopping
import sys
import scrapbook as sb

sys.path.append('..')
from utils import reshape_to_numpy, interpolate_missing_values, denoise_data, compute_spectrograms, time_avg_pooling

/home/marco/mnt/git/reckless-riding-detection/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
seed = 1
sampling_rate = 10
nperseg=64
noverlap=32
accel_cutoff = 0.4
accel_order = 30
gyro_cutoff = 0.6
gyro_order = 20
batch_size = 32
patience = 15
encoder_hidden_dim1 = 128
encoder_hidden_dim2 = 64
encoding_dim = 32
decoder_hidden_dim1 = 64
decoder_hidden_dim2 = 128
lr=0.001
n_splits = 5

In [3]:
# Parameters
seed = 7


In [4]:
rng = np.random.RandomState(seed)

In [5]:
data = pd.read_parquet("../data/GBG500.parquet")
data

,ride_id,time_index,ax,ay,az,rx,ry,rz
0,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,0,-2.512796,-9.385012,-1.053078,-0.009155,0.009155,-0.201416
1,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,100,-2.491268,-9.385012,-1.079390,-0.036621,0.027465,-0.155639
2,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,200,-2.534324,-9.382022,-1.030952,0.036621,0.027465,-0.073242
3,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,300,-2.488278,-9.383218,-1.091948,-0.045776,0.036621,-0.183105
4,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,400,-2.483494,-9.382022,-1.100320,-0.045776,0.027465,-0.192260
...,...,...,...,...,...,...,...,...
1529542,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241300,0.459862,-9.140430,-3.318900,1.556396,-0.091552,0.274658
1529543,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241400,0.455676,-9.140430,-3.321292,1.583861,-0.119018,0.274658
1529544,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241500,0.459862,-9.139832,-3.317704,1.583861,-0.109863,0.274658
1529545,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241600,0.456274,-9.142224,-3.321292,1.574707,-0.137329,0.283813


In [6]:
labels = pd.read_csv("../data/GBG500_labels.csv")
labels.columns = labels.columns.str.lower()
ride_order_df = pd.DataFrame({"ride_id": data["ride_id"].unique()})
labels_sorted = ride_order_df.merge(
    labels,
    on="ride_id",
    how="left"
)
labels_sorted

,ride_id,label
0,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,Safe
1,00d223cb7aecc9c0cc5871e32a6c45027a068eba07c572...,Reckless
2,02288a4aeca044203394e982e76b87818021dea2b34df9...,Safe
3,0236bdcf13d473ea24d97f5eaeea459f257dffac005694...,Bad weather
4,0238e5dd85143f1b6c59b200bc32f14361b8fc84c51a87...,Safe
...,...,...
495,fe5d7f84d692bbccad8bd566a3ad5c3108fa9f90acd671...,Safe
496,fe67169632306d4668b2affedef510df2cb43e2fb41220...,Safe
497,fee44667fdc8ab4995c702fc3bd36178de0b1ca2c64687...,Safe
498,ff79b2e945b93c2a5efc3a06365267b3a160e7849ba57d...,Safe


In [7]:
data_np = reshape_to_numpy(
    data,
    features = ["ax", "ay", "az", "rx", "ry", "rz"],
    max_timestamps = 4800
)

data_np_clean = interpolate_missing_values(
    data_np,
    method='linear',
    limit=None
)

data_np_clean = denoise_data(
    data=data_np_clean,
    accel_indices=[0, 1, 2],
    gyro_indices=[3, 4, 5],
    accel_cutoff=accel_cutoff,
    accel_order=accel_order,
    gyro_cutoff=gyro_cutoff,
    gyro_order=gyro_order,
)

In [8]:
# Compute spectrograms for all rides
spectrograms_array = compute_spectrograms(data_np_clean, sampling_rate, nperseg, noverlap)
spectrograms_array.shape

(500, 149, 6, 33)

In [9]:
# Aggregate spectrograms using time-averaged pooling
X_feat = time_avg_pooling(spectrograms_array)
X_feat.shape

(500, 198)

In [10]:
# Define Autoencoder architecture with Lightning
class Autoencoder(L.LightningModule):
    def __init__(
        self,
        input_dim,
        encoding_dim,
        encoder_hidden_dim1,
        encoder_hidden_dim2,
        decoder_hidden_dim1,
        decoder_hidden_dim2,
        lr,
    ):
        super().__init__()
        self.save_hyperparameters()
        
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, encoder_hidden_dim1),
            nn.ReLU(),
            nn.Linear(encoder_hidden_dim1, encoder_hidden_dim2),
            nn.ReLU(),
            nn.Linear(encoder_hidden_dim2, encoding_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(encoding_dim, decoder_hidden_dim1),
            nn.ReLU(),
            nn.Linear(decoder_hidden_dim1, decoder_hidden_dim2),
            nn.ReLU(),
            nn.Linear(decoder_hidden_dim2, input_dim)
        )
        self.criterion = nn.MSELoss()
    
    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded
    
    def encode(self, x):
        return self.encoder(x)
    
    def training_step(self, batch, batch_idx):
        x, _ = batch
        x_hat = self.forward(x)
        loss = self.criterion(x_hat, x)
        self.log('train_loss', loss, prog_bar=True)
        return loss
    
    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)

In [11]:
# 5-fold stratified CV with AE
y_true = (labels_sorted['label'] == 'Reckless').astype(int).values
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=rng.randint(1000))
scores = {k: np.full(len(y_true), np.nan) for k in ['lof', 'iso_forest', 'ocsvm', 'rec_err']}

for fold, (train_idx, test_idx) in enumerate(skf.split(X_feat, y_true)):
    fold_scaler = StandardScaler()
    X_train = fold_scaler.fit_transform(X_feat[train_idx])
    X_test = fold_scaler.transform(X_feat[test_idx])

    L.seed_everything(rng.randint(1000))
    autoencoder = Autoencoder(
        input_dim=X_train.shape[1],
        encoding_dim=encoding_dim,
        encoder_hidden_dim1=encoder_hidden_dim1,
        encoder_hidden_dim2=encoder_hidden_dim2,
        decoder_hidden_dim1=decoder_hidden_dim1,
        decoder_hidden_dim2=decoder_hidden_dim2,
        lr=lr,
    )

    X_train_tensor = torch.FloatTensor(X_train)
    dataset = TensorDataset(X_train_tensor, X_train_tensor)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    state = {'best_ewm_loss': float('inf'), 'best_state': None, 'ewm': None}

    class FoldLossTracker(L.pytorch.callbacks.Callback):
        def on_train_epoch_end(self, trainer, pl_module):
            loss = float(trainer.callback_metrics['train_loss'])
            if state['ewm'] is None:
                state['ewm'] = loss
            else:
                state['ewm'] = 0.2 * loss + 0.8 * state['ewm']
            if state['ewm'] < state['best_ewm_loss']:
                state['best_ewm_loss'] = state['ewm']
                state['best_state'] = copy.deepcopy(pl_module.state_dict())
            pl_module.log('train_loss_ewm_avg', state['ewm'])

    trainer = L.Trainer(
        max_epochs=300, accelerator='auto', devices=1,
        callbacks=[
            EarlyStopping(monitor='train_loss_ewm_avg', patience=patience,
                          verbose=False, mode='min', check_on_train_epoch_end=True),
            FoldLossTracker(),
        ],
        enable_progress_bar=False,
    )
    trainer.fit(autoencoder, dataloader)

    if state['best_state'] is not None:
        autoencoder.load_state_dict(state['best_state'])

    autoencoder.eval()
    X_test_tensor = torch.FloatTensor(X_test)
    with torch.no_grad():
        Z_train = autoencoder.encode(X_train_tensor).numpy()
        Z_test = autoencoder.encode(X_test_tensor).numpy()
        X_test_rec = autoencoder(X_test_tensor).numpy()

    enc_scaler = StandardScaler()
    Z_train = enc_scaler.fit_transform(Z_train)
    Z_test = enc_scaler.transform(Z_test)

    lof = LocalOutlierFactor(n_neighbors=20, novelty=True)
    lof.fit(Z_train)
    scores['lof'][test_idx] = -lof.score_samples(Z_test)

    iso = IsolationForest(random_state=rng.randint(1000))
    iso.fit(Z_train)
    scores['iso_forest'][test_idx] = -iso.score_samples(Z_test)

    ocsvm = OneClassSVM(kernel='rbf')
    ocsvm.fit(Z_train)
    scores['ocsvm'][test_idx] = -ocsvm.decision_function(Z_test)

    scores['rec_err'][test_idx] = np.mean((X_test - X_test_rec) ** 2, axis=1)

    print(f"Fold {fold+1}/{n_splits} done")

Seed set to 196


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


GPU available: True (cuda), used: True


TPU available: False, using: 0 TPU cores


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


...

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]



  | Name      | Type       | Params | Mode 
-------------------------------------------------
0 | encoder   | Sequential | 35.8 K | train
1 | decoder   | Sequential | 36.0 K | train
2 | criterion | MSELoss    | 0      | train
-------------------------------------------------
71.8 K    Trainable params
0         Non-trainable params
71.8 K    Total params
0.287     Total estimated model params size (MB)
13        Modules in train mode
0         Modules in eval mode


Fold 4/5 done


/home/marco/mnt/git/reckless-riding-detection/.venv/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:433: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.
/home/marco/mnt/git/reckless-riding-detection/.venv/lib/python3.10/site-packages/lightning/pytorch/loops/fit_loop.py:310: The number of training batches (13) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


Fold 5/5 done


In [12]:
for key, glue_name in [('lof', 'GBG500_ap_spectral_ae_lof'),
                        ('iso_forest', 'GBG500_ap_spectral_ae_iso_forest'),
                        ('ocsvm', 'GBG500_ap_spectral_ae_ocsvm'),
                        ('rec_err', 'GBG500_ap_spectral_ae_rec_err')]:
    ap = average_precision_score(y_true, scores[key])
    print(f"AE+{key} AP = {ap:.4f}")
    sb.glue(glue_name, float(ap))

AE+lof AP = 0.7774


AE+iso_forest AP = 0.5893


AE+ocsvm AP = 0.5272


AE+rec_err AP = 0.6018
